# NIDS ETL Pipeline: Feature Extraction from PCAP

Este notebook demuestra el proceso de extracción de características a nivel de ventana para el modelo L1 (XGBoost) y de secuencias para el modelo L2 (Transformer), directamente desde los archivos PCAP crudos.

In [ ]:
import os
import pickle
import sys

# Añadir el directorio raíz al path para importar src
sys.path.append(os.path.abspath('..'))

from src.ai.feature_eng import (
    calculate_window_stats, 
    tokenize_domain, 
    process_pcap
)

## Configuración de Parámetros
Definimos los parámetros globales para el windowing y los directorios de entrada/salida.

In [ ]:
RAW_DATA_DIR = '../data/raw/'
PROCESSED_DATA_DIR = '../data/processed/'
MAX_PACKETS_PER_WINDOW = 15 
TIMEOUT_SEC = 7.0 
L2_SEQ_LENGTH = 100 # Longitud máxima para tokenización de dominio

## Proceso ETL Principal
Iteramos sobre todos los archivos `.pcap` en la carpeta `data/raw/` (incluyendo `normal`, `tunnel`, etc.). Utilizamos la función modularizada `process_pcap` que agrupa los paquetes por `source_ip` en ventanas basándose en nuestros parámetros.

In [ ]:
def run_etl_pipeline():
    os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
    dataset = []
    
    LIMIT_FILES = 131
    file_count = 0

    for root, dirs, files in os.walk(RAW_DATA_DIR):
        for file in files:
            if not file.endswith('.pcap'):
                continue
                
            if file_count >= LIMIT_FILES:
                break
            
            pcap_path = os.path.join(root, file)
            folder_name = os.path.basename(root)
            
            # Etiquetado: 0 si proviene de la carpeta 'normal', 1 (Attack) en caso contrario
            label = 0 if folder_name == 'normal' else 1
            
            # 1. Agrupar por ventanas (Windows) usando scapy
            windows = process_pcap(pcap_path, 
                                   max_packets_per_window=MAX_PACKETS_PER_WINDOW, 
                                   timeout_sec=TIMEOUT_SEC)
            
            # 2. Generar características L1 y L2 por cada ventana
            for w in windows:
                l1_stats = calculate_window_stats(w)
                l2_seq = [tokenize_domain(pkt['domain'], max_len=L2_SEQ_LENGTH) for pkt in w]
                
                # Padding para mantener secuencias L2 de tamaño fijo (MAX_PACKETS_PER_WINDOW, L2_SEQ_LENGTH)
                while len(l2_seq) < MAX_PACKETS_PER_WINDOW:
                    l2_seq.append([0] * L2_SEQ_LENGTH)
                    
                dataset.append({
                    'l1_features': l1_stats,
                    'l2_sequence': l2_seq,
                    'label': label,
                    'source': pcap_path
                })
            
            file_count += 1
        if file_count >= LIMIT_FILES:
            break
            
    return dataset

# Ejecutar la canalización
dataset = run_etl_pipeline()
print(f"\nTotal de ventanas extraídas: {len(dataset)}")

## Exportar Datos Procesados
Guardamos los resultados de las ventanas generadas en formato `.pkl` para su posterior ingestión en el entrenamiento de los modelos XGBoost y Transformer.

In [ ]:
output_path = os.path.join(PROCESSED_DATA_DIR, 'nids_dataset.pkl')
with open(output_path, 'wb') as f:
    pickle.dump(dataset, f)
    
print(f"Dataset guardado exitosamente en: {output_path}")

## Exploratory Data Analysis (EDA)
A continuación, analizaremos las características estadísticas de capa L1 extraídas en las ventanas para observar cómo se diferencian los paquetes benignos de los túneles DNS.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

# Cargar las características L1 en un DataFrame para EDA
df_l1 = pd.DataFrame([d['l1_features'] for d in dataset])
df_l1['label'] = [d['label'] for d in dataset]

# Mostrar las primeras filas

In [ ]:
# Estadísticas descriptivas
display(df_l1.groupby('label').describe().T)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df_l1, x='label', hue='label', palette=['#2980b9','#c0392b'], legend=False)
plt.title('Distribución de Clases (0 = Benign, 1 = Attack)')

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.boxplot(data=df_l1, x='label', y='entropy_mean', hue='label', palette=['#2980b9','#c0392b'], legend=False)
plt.title('Entropía Promedio por Ventana')

plt.subplot(1, 2, 2)
sns.boxplot(data=df_l1, x='label', y='bytes_sum', hue='label', palette=['#2980b9','#c0392b'], legend=False)
plt.title('Suma de Bytes por Ventana')
plt.tight_layout()